# M8e · Final ML Dataset Builder

**Goal:** consolidate the four preceding steps into a single reproducible
`scikit-learn` `Pipeline` that can be re-run end-to-end on fresh marts and
produces the four canonical artifacts modelling consumes.

**Pipeline stages:**

1. **Source** — `marts.v_ml_features_full`
2. **Imputation** — `SimpleImputer(strategy='median')` for numeric;
   conceptual zero-fills hard-coded.
3. **Outlier handling** — `FunctionTransformer` winsorizing to [P1, P99].
4. **Encoding** — `ColumnTransformer` with OneHot for nominal,
   `OrdinalEncoder` for ordered, target-guided for high-card (skipped here).
5. **Split + SMOTE** — 80/20 stratified split, SMOTE on train only.

**Outputs:** `X_train.parquet`, `X_test.parquet`, `y_train.parquet`,
`y_test.parquet` plus a serialized preprocessing pipeline
(`preprocess.joblib`) so Phase 6 (deployment) can apply identical
transformations to incoming data.

> **This notebook is the M8 milestone exit gate.** Once it runs green,
> Phase 4 (Modelling) is unblocked.


In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


## 2. Build the preprocessing pipeline

In [ ]:
import pandas as pd, numpy as np, pathlib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import joblib

# ---------------------------------------------------------------------------
# 2a. Pull the raw feature matrix + categoricals
# ---------------------------------------------------------------------------
with get_engine().connect() as conn:
    df = pd.read_sql(text('SELECT * FROM marts.v_ml_features_full'), conn)
    dims = pd.read_sql(text('''
        SELECT dd.tenant_id, dd.device_id,
               dv.make AS vehicle_make,
               dv.vehicle_class
          FROM warehouse.dim_device dd
          JOIN warehouse.dim_vehicle dv ON dv.vehicle_sk = dd.vehicle_sk
    '''), conn)
df = df.merge(dims, on=['tenant_id', 'device_id'], how='left')
df[['vehicle_make', 'vehicle_class']] = df[['vehicle_make', 'vehicle_class']].fillna('unknown')
print('raw matrix:', df.shape)


## 3. Define the target and split off identity columns

In [ ]:
ELIGIBLE = df['total_distance_km'] >= 100
threshold = df.loc[ELIGIBLE, 'harsh_events_per_100km'].quantile(0.75)
df['is_high_risk'] = ((df['harsh_events_per_100km'] >= threshold) & ELIGIBLE).astype(int)

ID_COLS    = ['tenant_id', 'device_id', 'year_month']
LEAK_COLS  = ['harsh_events_per_100km', 'harsh_event_total',
              'harsh_brake_count', 'harsh_accel_count', 'harsh_corner_count']
TARGET     = 'is_high_risk'

X = df.drop(columns=ID_COLS + LEAK_COLS + [TARGET])
y = df[TARGET]
print('X cols:', X.shape[1])


## 4. Compose the ColumnTransformer (impute + winsorize + encode)

In [ ]:
NOMINAL = ['vehicle_make']
ORDINAL = ['vehicle_class']
ORDINAL_CATS = [['unknown', 'light', 'medium', 'heavy']]
NUMERIC = [c for c in X.columns if c not in NOMINAL + ORDINAL]

def winsorize(arr: np.ndarray) -> np.ndarray:
    a = np.asarray(arr, dtype=float)
    lo = np.nanpercentile(a, 1, axis=0)
    hi = np.nanpercentile(a, 99, axis=0)
    return np.clip(a, lo, hi)

numeric_pipe = Pipeline([
    ('imputer',    SimpleImputer(strategy='median')),
    ('winsorize',  FunctionTransformer(winsorize, validate=False)),
])

preprocess = ColumnTransformer([
    ('num', numeric_pipe, NUMERIC),
    ('ord', OrdinalEncoder(categories=ORDINAL_CATS,
                            handle_unknown='use_encoded_value',
                            unknown_value=-1), ORDINAL),
    ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), NOMINAL),
])
preprocess


## 5. Split, fit, transform, SMOTE

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train_t = preprocess.fit_transform(X_train_raw)
X_test_t  = preprocess.transform(X_test_raw)

# Reconstruct column names after ColumnTransformer
ohe = preprocess.named_transformers_['ohe']
feature_names = (NUMERIC
                 + ORDINAL
                 + [f'make_{c}' for c in ohe.categories_[0]])
X_train_t = pd.DataFrame(X_train_t, columns=feature_names)
X_test_t  = pd.DataFrame(X_test_t,  columns=feature_names)

sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train_t, y_train)
print('train (balanced):', X_train_bal.shape, 'test:', X_test_t.shape)


## 6. Persist artifacts

In [ ]:
out = pathlib.Path('../../data/ml')
out.mkdir(parents=True, exist_ok=True)
X_train_bal.to_parquet(out / 'X_train.parquet', index=False)
X_test_t.to_parquet(out / 'X_test.parquet', index=False)
pd.DataFrame({'is_high_risk': y_train_bal}).to_parquet(out / 'y_train.parquet', index=False)
pd.DataFrame({'is_high_risk': y_test}).to_parquet(out / 'y_test.parquet', index=False)
joblib.dump(preprocess, out / 'preprocess.joblib')
print('artifacts:', sorted(p.name for p in out.iterdir()))


## 7. Sanity / exit checks

In [ ]:
import joblib
loaded = joblib.load(out / 'preprocess.joblib')
sample = X_test_raw.head(3)
print('round-trip transform shape:', loaded.transform(sample).shape)
print('OK — Phase 4 (Modelling) is unblocked.')
